# Contextual Compression Retriever — 맥락 기반 문서 압축 검색기

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **ContextualCompressionRetriever**의 작동 원리 이해하기
- `LLMChainExtractor`, `LLMChainFilter`, `EmbeddingsFilter` 각각의 차이 파악하기
- `DocumentCompressorPipeline`으로 여러 압축 단계를 조합하기

## 사전 지식

- 01-VectorStoreRetriever에서 기본 Retriever를 만들어본 경험
- LLM(대규모 언어 모델)과 임베딩 모델의 역할 이해

---

```mermaid
flowchart LR
    Q[사용자 쿼리]:::input --> BR[Base Retriever<br/>후보 문서 검색]:::process
    BR --> C{Compressor<br/>선택}:::process
    C -- LLMExtractor --> E1[LLM이<br/>관련 내용 추출]:::process
    C -- LLMFilter --> E2[LLM이<br/>문서 선별]:::process
    C -- EmbeddingsFilter --> E3[임베딩<br/>유사도 필터]:::process
    E1 --> O[압축된<br/>문서]:::output
    E2 --> O
    E3 --> O

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## 핵심 문제

일반 Retriever는 문서 전체를 반환하지만, 실제로 필요한 정보는 문서의 일부분일 수 있어요:

- **문제 1**: 무관한 텍스트가 많이 포함되어 LLM 비용 증가
- **문제 2**: 노이즈가 많아 응답 품질 저하
- **문제 3**: 컨텍스트 길이 제한으로 중요 정보 누락

## 해결 방법

`ContextualCompressionRetriever`는 두 단계로 작동해요:

1. **검색(Retrieval)**: Base Retriever로 후보 문서 검색
2. **압축(Compression)**: Document Compressor로 관련 내용만 추출

## 환경 설정

In [ ]:
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import (

# 아래에 코드를 작성하세요


## 1. 기본 Retriever 준비

먼저 압축하지 않은 기본 retriever의 결과를 확인해봅시다.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------------------------------------------------
# 압축 기법 비교를 위한 기본 Retriever 준비
# ---------------------------------------------------

# ============================================================
# TODO: ai-story.txt를 로드하고 800자 청크로 분할한 뒤 FAISS Retriever를 생성하세요
# 힌트: chunk_size=800으로 크게 설정해야 압축 효과가 잘 보임
# 예상 결과: 총 청크 수와 기본 Retriever 생성 완료 메시지 출력
# ============================================================

# 문서 로드 및 분할

# 큰 청크를 사용해야 압축 후 크기 감소 효과가 뚜렷이 보임

# 벡터 데이터베이스 생성

# 기본 retriever 생성


In [ ]:
# 헬퍼 함수: 문서를 깔끔하게 출력
def print_docs(docs, title="검색 결과"):
    print(f"\n{'='*80}")
    print(f"{title}")
    print(f"{'='*80}\n")
    
    for i, doc in enumerate(docs, 1):
        print(f"[문서 {i}]")
        print(f"내용: {doc.page_content}")
        print(f"길이: {len(doc.page_content)} 문자")
        print("-" * 80 + "\n")


In [ ]:
# ---------------------------------------------------
# 기본 Retriever 결과를 확인해 압축 전 상태 파악
# ---------------------------------------------------

# ============================================================
# TODO: base_retriever로 쿼리를 검색하고 총 문자 수를 계산하세요
# 힌트: retriever.invoke(query) 후 sum(len(d.page_content) for d in docs)
# 예상 결과: 원본 크기 확인 (압축 후와 비교할 기준선)
# ============================================================

# 테스트 쿼리

# 기본 retriever로 검색


# 전체 문자 수 계산


## 2. LLMChainExtractor - LLM 기반 내용 추출

`LLMChainExtractor`는 LLM을 사용하여 쿼리와 관련된 내용만 추출합니다.

### 작동 방식

1. 검색된 각 문서에 대해 LLM에게 질문
2. "이 문서에서 쿼리와 관련된 부분만 추출하세요"
3. LLM이 관련 내용만 반환

### 장점과 단점

**장점:**
- 매우 정확한 내용 추출
- 문맥을 이해하여 관련성 판단

**단점:**
- LLM 호출 비용 발생
- 속도가 느림 (각 문서마다 LLM 호출)

In [ ]:
from langchain_openai import ChatOpenAI

# LLM 초기화

# LLMChainExtractor 생성

# ContextualCompressionRetriever 생성

# 아래에 코드를 작성하세요


In [ ]:
# 압축된 검색 실행


# 압축 효과 비교

# 아래에 코드를 작성하세요


## 3. LLMChainFilter - LLM 기반 문서 필터링

`LLMChainFilter`는 문서 내용을 압축하지 않고, 관련 있는 문서만 선택적으로 반환합니다.

### LLMChainExtractor vs LLMChainFilter

| 특징 | LLMChainExtractor | LLMChainFilter |
|------|------------------|----------------|
| **동작** | 문서 내용 추출/압축 | 문서 전체 유지 또는 제거 |
| **결과** | 압축된 내용 반환 | 원본 문서 또는 빈 결과 |
| **용도** | 긴 문서에서 핵심 추출 | 관련 없는 문서 제거 |
| **속도** | 느림 | 느림 |
| **비용** | 높음 (추출 작업) | 높음 (판단 작업) |

In [ ]:
# ---------------------------------------------------
# LLMChainFilter로 관련 없는 문서 제거 (내용 압축 없이)
# ---------------------------------------------------

# ============================================================
# TODO: LLMChainFilter를 사용해 관련 없는 문서를 제거하는 retriever를 만들고 검색하세요
# 힌트: LLMChainFilter.from_llm(llm) → ContextualCompressionRetriever
# 예상 결과: 원본 문서를 그대로 유지하되 관련 없는 문서는 제거됨
# ============================================================

# LLMChainFilter 생성
# LLMChainFilter: 문서 내용을 압축하지 않고 관련/비관련 이진 판단

# ContextualCompressionRetriever 생성

# 필터링된 검색 실행


## 4. EmbeddingsFilter - 임베딩 기반 필터링

`EmbeddingsFilter`는 임베딩 유사도를 사용하여 빠르고 저렴하게 문서를 필터링합니다.

### 장점

- **빠른 속도**: LLM 호출 없이 벡터 연산만 수행
- **낮은 비용**: 임베딩 유사도 계산만 필요
- **효율성**: 대량의 문서 필터링에 적합

### 언제 사용하면 좋을까요?

- 빠른 응답이 필요한 애플리케이션
- 비용을 최소화해야 할 때
- 이미 검색된 문서를 추가로 필터링할 때

> **실무 팁**: 프로덕션 환경에서 Compression Retriever가 필요하다면 EmbeddingsFilter를 먼저 선택하세요. LLM 호출 없이 동작하므로 레이턴시 증가가 거의 없고, 비용도 절감됩니다.

In [ ]:
# EmbeddingsFilter 생성 (유사도 임계값 0.75)

# ContextualCompressionRetriever 생성

# 임베딩 기반 필터링 실행

# 아래에 코드를 작성하세요


## 5. DocumentCompressorPipeline - 여러 Compressor 조합

`DocumentCompressorPipeline`을 사용하면 여러 compressor와 transformer를 순차적으로 적용할 수 있습니다.

### 파이프라인 구성 요소

1. **TextSplitter**: 문서를 더 작은 청크로 분할
2. **EmbeddingsRedundantFilter**: 중복 문서 제거 (유사도 0.95 이상)
3. **EmbeddingsFilter**: 관련성 필터링 (유사도 임계값)
4. **LLMChainExtractor**: 최종 내용 추출

### 파이프라인 전략

```
큰 문서 → 작은 청크 → 중복 제거 → 관련성 필터링 → 내용 추출
```

> **실무 팁**: 파이프라인 단계 순서가 중요합니다. 비용이 낮은 단계(TextSplitter, EmbeddingsFilter)를 먼저 적용해 문서를 줄이고, 비용이 높은 단계(LLMChainExtractor)는 마지막에 적용하세요.

In [ ]:
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_text_splitters import CharacterTextSplitter

# 1. 텍스트 분할기 (문서를 더 작은 청크로)

# 2. 중복 필터 (유사도 0.95 이상인 문서 제거)

# 3. 관련성 필터 (유사도 0.75 이상만 유지)

# 4. LLM 내용 추출기

# 파이프라인 생성

# 아래에 코드를 작성하세요


In [ ]:
# ---------------------------------------------------
# 파이프라인 적용 후 모든 방식 비교
# ---------------------------------------------------

# ============================================================
# TODO: pipeline_retriever로 검색하고 4가지 방식의 문서 수와 총 문자 수를 비교하세요
# 힌트: pipeline_retriever.invoke(query) 후 비교표 출력
# 예상 결과: Pipeline이 가장 압축된 결과를 반환
# ============================================================

# 파이프라인 적용

# 파이프라인 실행


# 전체 비교


## 6. Compressor 선택 가이드

### 사용 시나리오별 추천

| 시나리오 | 추천 Compressor | 이유 |
|---------|----------------|------|
| **최고 품질 필요** | LLMChainExtractor | 가장 정확한 내용 추출 |
| **빠른 응답 필요** | EmbeddingsFilter | LLM 호출 없음 |
| **비용 최소화** | EmbeddingsFilter | 임베딩만 사용 |
| **문서 선택만** | LLMChainFilter | 원본 유지하며 필터링 |
| **최적 균형** | Pipeline | 단계별 최적화 |

### 비용 vs 품질 트레이드오프

```
높은 품질 ← → 낮은 비용/빠른 속도

LLMChainExtractor > Pipeline > LLMChainFilter > EmbeddingsFilter
```

### 실전 권장 조합

```python
# Production 환경 권장
pipeline = DocumentCompressorPipeline([
    CharacterTextSplitter(chunk_size=400),  # 작은 청크
    EmbeddingsRedundantFilter(),           # 중복 제거
    EmbeddingsFilter(threshold=0.75),      # 빠른 필터링
    # LLMChainExtractor()  # 선택적: 최종 품질 향상
])
```

## 마무리 정리

이 노트북에서 배운 내용을 정리해요:

| Compressor | 비용 | 속도 | 품질 | 추천 상황 |
|-----------|------|------|------|----------|
| **LLMChainExtractor** | 높음 | 느림 | 최고 | 품질이 최우선 |
| **LLMChainFilter** | 높음 | 느림 | 높음 | 원본 유지하며 필터링 |
| **EmbeddingsFilter** | 낮음 | 빠름 | 보통 | 비용·속도 중시 |
| **Pipeline** | 중간 | 중간 | 높음 | 균형 잡힌 운영 |

### 다음 노트북 예고

**03-EnsembleRetriever**: BM25 키워드 검색과 벡터 검색을 결합하는 하이브리드 검색을 배워요. 검색 방식의 약점을 서로 보완하는 실무 필수 기법입니다.